# Demo: GOES VINTAGE

This notebook takes a user-defined lat/lon box, date range, and UTC time-of-day range, then runs the full workflow: download GOES, orthorectify, build Zarr, build RGB, apply the ERA5-Land-temperature-dependent GOES VINTAGE mask, and plot RGB next to the VINTAGE mask.

**Important:** the GOES VINTAGE method is intended for daylight imagery only. Choose UTC hours when your full domain is sunlit. The default Colorado demo uses one 5-minute scan from `20 UTC` on `2020-06-30`, so it downloads only three GOES files total: one each for `C02`, `C05`, and `C13`. Download hour ranges are inclusive, so `18-22` means five hours.

## 1. Set Your Output Label, Bounds, Dates, And Daylight Hours

Edit these values first. DOMAIN is only an output filename label; the latitude/longitude bounds below define the actual processing area. Longitudes west of Greenwich should be negative.

In [ ]:
from pathlib import Path

# Default test case: Colorado output label and bounds, June 27, 2020
# DOMAIN is only used as an output filename suffix.
DOMAIN = "colorado"
# check which GOES satellite was operational for your time period and whether you want
# GOES-East or GOES-West depending on your domain
GOES = "goes16"
START_DATE = "2020-06-27"
END_DATE = "2020-06-27"

# Bounding box: lon_min, lat_min, lon_max, lat_max
LON_MIN = -109.0
LAT_MIN = 37.0
LON_MAX = -104.0
LAT_MAX = 41.0

# Daylight UTC window. The mask is not designed for nighttime imagery.
# GOES_HOURS controls both downloaded hours and the mask time window.
# It accepts a single hour ("20"), inclusive ranges ("18-22"), or comma lists.
# GOES_TIMESTEPS_PER_HOUR=1 keeps only one 5-minute scan per channel for a fast demo.
# Set GOES_TIMESTEPS_PER_HOUR=None to download every scan in each selected hour.
GOES_HOURS = "20"
GOES_TIMESTEPS_PER_HOUR = 1

# Keep outputs somewhere with enough space.
BASE_DIR = Path("./demo_output/colorado")

# Set False to preview the commands without downloading/processing data.
RUN_WORKFLOW = True
OVERWRITE_MASK = True
KEEP_MASK_DIAGNOSTICS = False

# Time shown in the final plot. None uses the first available mask timestep.
PLOT_TIME_UTC = None
PLOT_PATH = None

## 2. Build The Workflow Config

In [ ]:
import os
import shlex
import sys
from pathlib import Path

REPO_DIR = Path.cwd()
if not (REPO_DIR / "scripts").exists() and (REPO_DIR.parent / "scripts").exists():
    REPO_DIR = REPO_DIR.parent
SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

PRIVATE_ENV = REPO_DIR / "private_env.sh"
if PRIVATE_ENV.exists():
    for line in PRIVATE_ENV.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export " ):]
        if "=" not in line:
            continue
        name, value = line.split("=", 1)
        os.environ[name.strip()] = shlex.split(value, comments=False, posix=True)[0]

import workflow as wf

BASE_DIR = BASE_DIR if BASE_DIR.is_absolute() else REPO_DIR / BASE_DIR

config = wf.WorkflowConfig(
    domain=DOMAIN,
    goes=GOES,
    start_date=START_DATE,
    end_date=END_DATE,
    lon_min=LON_MIN,
    lat_min=LAT_MIN,
    lon_max=LON_MAX,
    lat_max=LAT_MAX,
    base_dir=BASE_DIR,
    goes_hours=GOES_HOURS,
    goes_timesteps_per_hour=GOES_TIMESTEPS_PER_HOUR,
    overwrite_mask=OVERWRITE_MASK,
    keep_mask_diagnostics=KEEP_MASK_DIAGNOSTICS,
    run=RUN_WORKFLOW,
)

print(f"Workflow dates: {config.start_date} to {config.end_date}")
print(f"Bounds: {LON_MIN}, {LAT_MIN}, {LON_MAX}, {LAT_MAX}")
print(f"Daylight UTC window: {config.start_hour_utc:g}-{config.end_hour_utc:g}")
print(f"GOES timesteps per hour: {GOES_TIMESTEPS_PER_HOUR}")
print(f"Output base: {config.base_dir.resolve()}")

## 3. Check Credentials

GOES data is easily accessible via AWS. OpenTopography and Copernicus/CDS require your own keys. OpenTopography is used to get a DEM for the orthorectification process. See the goes-ortho python package and Pestana & Lundquist (2022). The Copernicus API key is used for ERA5-Land surface temperature reanalysis for the temperature-dependent VINTAGE mask thresholds.


Pestana, S., & Lundquist, J. D. (2022). Evaluating GOES-16 ABI surface brightness temperature observation biases over the central Sierra Nevada of California. Remote Sensing of Environment, 281, 113221.

https://github.com/spestana/goes-ortho

In [ ]:
wf.validate_credentials(config)

## 4. Run The Workflow

Each step is a one-line function. Progress bars show where the workflow is, and noisy command output is hidden unless a command fails.

In [ ]:
wf.download_goes(config)

In [ ]:
wf.orthorectify(config)

In [ ]:
wf.build_zarr(config)

In [ ]:
rgb_paths = wf.build_rgb(config)
rgb_paths

In [ ]:
mask_paths = wf.apply_mask(config)
mask_paths

## 5. Plot RGB And Mask

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

if not RUN_WORKFLOW:
    raise RuntimeError("Set RUN_WORKFLOW=True and run the workflow cells before plotting outputs.")

plot_date = pd.Timestamp(START_DATE)
rgb_path = config.rgb_path(plot_date)
mask_path = config.mask_path(plot_date)

with xr.open_dataset(rgb_path) as rgb_ds, xr.open_dataset(mask_path) as mask_ds:
    plot_time = pd.Timestamp(PLOT_TIME_UTC) if PLOT_TIME_UTC is not None else pd.Timestamp(mask_ds["t"].values[0])
    rgb_frame = rgb_ds.sel(t=plot_time, method="nearest")
    mask_frame = mask_ds.sel(t=plot_time, method="nearest")
    actual_time = pd.Timestamp(mask_frame["t"].values)

    rgb_image = np.stack(
        [rgb_frame["red"].values, rgb_frame["green"].values, rgb_frame["blue"].values],
        axis=-1,
    )
    rgb_image = np.clip(np.nan_to_num(rgb_image, nan=0.0), 0.0, 1.0)
    vintage_mask = mask_frame["vintage_mask"].values
    lon = rgb_ds["longitude"].values
    lat = rgb_ds["latitude"].values
    extent = [float(np.nanmin(lon)), float(np.nanmax(lon)), float(np.nanmin(lat)), float(np.nanmax(lat))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].imshow(rgb_image, origin="lower", extent=extent, aspect="auto")
axes[0].set_title(f"GOES DayCloudPhase RGB\n{actual_time:%Y-%m-%d %H:%M UTC}")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

im = axes[1].imshow(vintage_mask, origin="lower", extent=extent, aspect="auto", vmin=0, vmax=1, cmap="Blues_r")
axes[1].set_title("GOES VINTAGE Mask")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
cbar = fig.colorbar(im, ax=axes[1], ticks=[0, 1], shrink=0.8)
cbar.ax.set_yticklabels(["clear", "cloud"])
plot_path = Path(PLOT_PATH) if PLOT_PATH else config.base_dir / config.goes / "plots" / f"{config.goes}_rgb_mask_{config.domain}_{config.date_ymd(plot_date)}_notebook.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=160)
print(f"Wrote PNG: {plot_path}")
plt.show()